In [1]:
"""
Load a pretrained model and run inference on it.
For the double-encoder model
"""

import torch
# from double_train_fm import ConditionalFlowMatchingModule
from train_fm import ConditionalFlowMatchingModule


checkpoint_path = '/data/vision/billf/scratch/pablomer/projects/tess-generative/galaxy_images/galaxy_model/galaxy-flow-matching/epdlfvpg/checkpoints/epoch=123-step=46000.ckpt'

model = ConditionalFlowMatchingModule.load_from_checkpoint(checkpoint_path)

# Set the model to evaluation mode and disable gradient calculation for inference
model.eval()
torch.set_grad_enabled(False)

# Move model to appropriate device (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())

print('Loaded model succesfully')
print(f'Total model params {total_params:,}')
print(f'Using device: {device}')

# Create input tensors on the same device as the model
target_images = torch.rand(64, 4, 48, 48, device=device)
cond_images = torch.rand(64, 4, 48, 48, device=device)
# x = model.sample(cond_images, num_steps=100)

# print('Reconstruction shape', x.shape)


def compute_mse(self, target_image, cond_image):
    '''Compute reconstruction MSE on a batch of given images
    Args:
        target_iamge (B,C,H,W)
        cond_image (B,C,H,W)
    '''
    batch_size = target_image.shape[0]

    samples = self.sample(cond_image)

    error = target_image - samples

    return error

print('Trying the compute_mse function')

error = compute_mse(model,target_images, cond_images)

print(error.shape)


/data/vision/billf/scratch/pablomer/miniconda3/envs/py310-torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/vision/billf/scratch/pablomer/miniconda3/envs/py310-torch/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they

Loaded model succesfully
Total model params 148,815,236
Using device: cuda
Reconstruction shape torch.Size([64, 4, 48, 48])
Trying the compute_mse function
torch.Size([64, 4, 48, 48])


In [ ]:
def compute_mse(self, target_image, cond_image):
    '''Compute reconstruction MSE on a batch of given images
    Args:
        target_iamge (B,C,H,W)
        cond_image (B,C,H,W)
    '''
    batch_size = target_image.shape[0]

    samples = self.sample(cond_image)

    diff = target_image - samples

    mse_error = torch.mean(diff**2)
    
    return mse_error


In [2]:
# print('Trying the compute_mse function')

# error = compute_mse(model,target_images, cond_images)

print(error.shape)

torch.Size([64, 4, 48, 48])


In [10]:
error_mean = error.mean()
print(error_mean.shape)
error_mean

torch.Size([])


tensor(0.8120, device='cuda:0')

In [8]:
target_images.repeat((5,64,4,48,48))

print(target_images.shape)

OutOfMemoryError: CUDA out of memory. Tried to allocate 6480.00 GiB. GPU 0 has a total capacity of 31.73 GiB of which 30.70 GiB is free. Including non-PyTorch memory, this process has 1.03 GiB memory in use. Of the allocated memory 589.93 MiB is allocated by PyTorch, and 86.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)